# Data preprocessing

This notebook loads the raw ELIPA spreadsheets from `data/raw`, tidies each modality into a consistent sample × feature format, and saves cleaned tables to `data/processed`.

In [1]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer

# Add src to path so we can import our modules
sys.path.append(str(Path('..').resolve()))

from src.utils.paths import RAW_DIR, PROCESSED_DIR, ensure_dirs
from src.data.preprocessing import (
    normalize_participant_id, 
    normalize_timepoint, 
    make_sample_id,
    extract_participant_timepoint
)

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

ensure_dirs()


## Metabolomics

In [2]:
met_path = RAW_DIR / 'metabolomics.xlsx'
met_raw = pd.read_excel(met_path, sheet_name=0)

meta = (
    met_raw.iloc[:3]
    .set_index('Unnamed: 0')
    .T
    .rename(columns={'ID': 'participant_id', 'group': 'group', 'time': 'timepoint'})
)
meta['participant_id'] = meta['participant_id'].apply(normalize_participant_id)
meta['timepoint'] = meta['timepoint'].apply(normalize_timepoint)
meta['sample_id'] = [
    make_sample_id(pid, tp) for pid, tp in zip(meta['participant_id'], meta['timepoint'])
]

met_features = met_raw.iloc[3:].set_index('Unnamed: 0')
met_features = met_features.apply(pd.to_numeric, errors='coerce')
met_features_t = met_features.T

# FIX: Handle duplicate feature names by making them unique
# The raw data has duplicate metabolite names; pandas adds .1, .2 suffixes when transposing
# We need to deduplicate BEFORE creating the final dataframe
met_features_t.columns = pd.Index([
    f"{col}_{i}" if list(met_features_t.columns).count(col) > 1 and i > 0 else col
    for i, col in enumerate(met_features_t.columns)
])

metabolomics = pd.concat([meta, met_features_t], axis=1)

# Reorder columns to put metadata first
meta_cols = ['sample_id', 'participant_id', 'timepoint', 'group']
feature_cols = [c for c in met_features_t.columns]
metabolomics = metabolomics[meta_cols + feature_cols]

print(f"Metabolomics shape: {metabolomics.shape}")
print(f"Unique columns: {len(set(metabolomics.columns))}")
metabolomics.head()


Metabolomics shape: (237, 101)
Unique columns: 101


,sample_id,participant_id,timepoint,group,Acetylcarnosine,Aminobutyric acid betaine,Aminovaleric acid betaine,Arachidamide,Arginine,Behenamide,...,Unknown glycosidic compound MW 488.3363,Unknown glycosidic compound MW 542.2741,Unknown phospholipid MW 501.3187,Unknown phospholipid MW 511.3264_90,Unknown phospholipid MW 511.3264_91,Unknown phospholipid MW 672.2820,Unknown phospholipid MW 787.5482,Unknown phospholipid MW 819.6127,Urea,Uridine
101002,ELP_002_1,002,1,1.0,5583.098460,117374.0419,582962.4127,5.347010e+06,219513.9713,15267210.44,...,126792.52900,3.263987e+05,1280892.748,1170860.226,8.211774e+06,195506.3526,3408356.307,4.297378e+06,41523.01657,350926.7647
101004,ELP_004_1,004,1,1.0,4037.177015,119076.2431,291502.7191,1.027758e+07,221988.6601,23977484.29,...,69579.00852,6.432418e+05,2162733.829,1702943.099,1.164110e+07,266135.5579,6330327.496,1.594239e+07,35804.39459,406621.3714
101005,ELP_005_1,005,1,1.0,7133.361832,146682.2472,220044.4438,9.181427e+06,283766.1636,33296852.37,...,329680.34980,1.352630e+06,1367609.291,1345291.016,1.311010e+07,254456.0716,4283798.940,1.122549e+07,47236.65577,332443.3193
101006,ELP_006_1,006,1,1.0,4987.933552,117170.7218,353543.9943,1.160057e+07,390459.0587,27091597.22,...,232563.69760,7.212152e+05,1744755.847,1845773.170,1.008019e+07,261901.9817,3453231.083,1.338056e+07,52884.23572,189756.8904
101009,ELP_009_1,009,1,1.0,3699.165436,129816.4878,644387.5744,8.175153e+06,268999.4157,26428962.25,...,101682.07680,4.186638e+05,2288918.053,1322655.917,1.520648e+07,239298.1096,5640081.376,7.971029e+06,43867.16621,284835.8713


## Microbiota

In [3]:
micro_path = RAW_DIR / 'microbiota.xlsx'
micro_raw = pd.read_excel(micro_path, sheet_name=0)

micro_columns = micro_raw.iloc[0].tolist()
micro_columns[0] = 'sample_id'
micro_raw.columns = micro_columns
micro_raw = micro_raw.iloc[1:].reset_index(drop=True)

micro_raw = micro_raw.rename(columns={'id_nro': 'participant_id', 'Time': 'timepoint'})
micro_raw['participant_id'] = micro_raw['participant_id'].apply(normalize_participant_id)
micro_raw['timepoint'] = micro_raw['timepoint'].apply(normalize_timepoint)

micro_raw['sample_id'] = micro_raw['sample_id'].replace(r'^\s*$', pd.NA, regex=True)

generated_sample_ids = micro_raw.apply(
    lambda row: make_sample_id(row['participant_id'], row['timepoint']),
    axis=1
)

micro_raw['sample_id'] = micro_raw['sample_id'].fillna(generated_sample_ids)

meta_cols = [c for c in ['sample_id', 'participant_id', 'timepoint', 'Pair', 'group'] if c in micro_raw.columns]
feature_cols = [c for c in micro_raw.columns if c not in meta_cols]
micro_features = micro_raw[feature_cols].apply(pd.to_numeric, errors='coerce')
microbiota = pd.concat([micro_raw[meta_cols], micro_features], axis=1)
microbiota.head()


,sample_id,participant_id,timepoint,Pair,Maintenancediet,evenness,richness,diversity,cluster,response,...,UnculturedClostridialesI,UnculturedClostridialesII,UnculturedMollicutes_A,UnculturedSelenomonadaceae,Veillonella,Vibrio,Weissellaetrel,Wissellaetrel,Xanthomonadaceae,Yersiniaetrel
0,ELP_002_1,002,1,2,NaN,0.812882,104,25.122242,1,1.0,...,1207.421495,1474.248091,542.994641,101.473895,268.466374,167.728751,124.211725,47.419349,78.157259,158.375159
1,ELP_002_2,002,2,2,NaN,0.805527,90,33.718771,1,1.0,...,1211.952884,1368.989141,522.061418,223.893650,234.786902,155.473496,111.566464,39.053706,77.933203,156.313674
2,ELP_002_4,002,4,2,1.0,0.851403,200,66.630665,1,1.0,...,1798.011445,2391.426225,568.652420,76.415906,186.512986,181.162275,297.256856,84.460442,77.958643,166.645566
3,ELP_007_1,007,1,7,NaN,0.814268,101,45.352493,3,2.0,...,7782.200567,1441.419769,547.872172,40.817812,432.012404,163.232787,121.693579,80.741888,82.842766,162.883949
4,ELP_007_2,007,2,7,NaN,0.820218,104,47.901394,3,2.0,...,2521.107478,13407.167438,591.338029,46.082694,170.740199,193.411910,132.365039,60.463435,89.634834,174.734123


## Transcriptomics

In [4]:
trans_path = RAW_DIR / 'transcriptomics.xls'
trans_raw = pd.read_excel(trans_path, sheet_name=0)

sample_cols = [c for c in trans_raw.columns if str(c).startswith('ELP_')]
# Create unique feature IDs: GeneSymbol_ID
symbols = trans_raw['gene_name'].fillna('Unknown').astype(str)
trans_raw['feature_id'] = symbols + "_" + trans_raw['ID'].astype(str)

trans_features = trans_raw[['feature_id', *sample_cols]].set_index('feature_id')
trans_features = trans_features.apply(pd.to_numeric, errors='coerce')

transcriptomics = trans_features.T
transcriptomics.index.name = 'sample_id'
transcriptomics = transcriptomics.reset_index()

participant_ids = []
timepoints = []
for sid in transcriptomics['sample_id']:
    pid, tp = extract_participant_timepoint(sid)
    participant_ids.append(pid)
    timepoints.append(tp)

transcriptomics['participant_id'] = participant_ids
transcriptomics['timepoint'] = timepoints
feature_cols = [c for c in transcriptomics.columns if c not in ['sample_id', 'participant_id', 'timepoint']]
transcriptomics = transcriptomics[['sample_id', 'participant_id', 'timepoint', *feature_cols]]
transcriptomics.head()

/tmp/ipykernel_580321/944906583.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  trans_raw['feature_id'] = symbols + "_" + trans_raw['ID'].astype(str)


feature_id,sample_id,participant_id,timepoint,LOC100009676_100009676,AKT3_10000,MED6_10001,NR2E3_10002,SNORD116-1_100033413,SNORD116-2_100033414,SNORD116-4_100033416,...,PPP4R1_9989,CDC42_998,SLC12A6_9990,ROD1_9991,KCNE2_9992,DGCR2_9993,CASP8AP2_9994,SCO2_9997,CDH1_999,NAT1_9
0,ELP_069_1,069,1,127.200651,551.014608,173.807028,22.684132,16.310426,102.400353,13.416565,...,223.846484,243.298786,261.520079,470.094215,16.345908,187.163490,149.298703,85.570122,12.558107,20.265572
1,ELP_069_4,069,4,125.511195,552.602581,158.777917,18.223659,22.658508,97.590787,11.756745,...,244.060163,274.140757,394.942027,523.805892,15.883682,227.352910,129.321423,80.310608,19.191046,17.095851
2,ELP_029_1,029,1,133.374395,587.192001,221.359988,20.255420,20.327396,86.313332,9.424819,...,292.213496,248.743561,291.326229,625.885485,16.576993,175.514362,147.284349,73.951529,12.790842,22.543954
3,ELP_029_2,029,2,134.863257,697.066692,207.862491,20.498601,21.683577,83.816728,12.188063,...,287.403898,294.427013,331.651371,631.500209,21.918146,186.162907,140.054726,54.887763,14.177801,20.720532
4,ELP_029_4,029,4,154.879527,570.107070,197.843654,26.390451,21.755462,77.004605,9.437750,...,252.122420,244.041204,267.653713,491.963810,21.514978,204.689433,138.201131,71.308059,11.611656,19.149285


## Clinical questionnaires and weights

In [5]:
quest_path = RAW_DIR / 'questionnaires.xlsx'
questionnaires = pd.read_excel(quest_path, sheet_name=0)
questionnaires = questionnaires.rename(columns={'id_nro': 'participant_id'})
questionnaires['participant_id'] = questionnaires['participant_id'].apply(normalize_participant_id)

weights_path = RAW_DIR / 'weights.xlsx'
weights = pd.read_excel(weights_path, sheet_name=0)
weights = weights.rename(columns={'id_nro': 'participant_id'})
weights['participant_id'] = weights['participant_id'].apply(normalize_participant_id)

if 'weight change' not in weights.columns and {'weight4', 'weight2'}.issubset(weights.columns):
    weights['weight change'] = weights['weight4'] - weights['weight2']

weights
questionnaires.head().head()

,nro,participant_id,group,maintenancePhase_duration,sex,energylevel_WM,age,marital,houshold_adults,houshold_child,...,vegetables_91400.5_mean,cucumber_91410.5_mean,tomato_91420.5_mean,pepper_91430.5_mean,cabbages_91440.5_mean,onion_91450.5_mean,lettuce_91460.5_mean,greens_91470.5_mean,canned_91480.5_mean,other_vegetables_91490.5_mean
0,2,002,1,169,2,1400,42,2,2,3,...,359.225,55.75,93.25,0.000,45.000,6.725,8.500,0.000,0.0,150.00
1,4,004,1,168,2,1400,56,2,2,0,...,243.650,25.25,9.00,18.375,60.175,31.925,42.000,1.850,2.5,52.55
2,5,005,1,168,1,1800,62,4,2,0,...,174.050,6.00,6.00,10.500,74.425,7.700,25.500,0.675,2.5,40.75
3,6,006,1,169,1,1800,62,3,1,0,...,141.900,33.75,105.00,0.000,0.000,3.150,0.000,0.000,0.0,0.00
4,7,007,2,185,1,2200,34,2,2,0,...,113.275,36.55,47.15,2.225,0.000,2.425,24.725,0.225,0.0,0.00


## Alignment checks and saving cleaned tables

In [6]:
met_ids = set(metabolomics['sample_id'].dropna())
micro_ids = set(microbiota['sample_id'].dropna())
trans_ids = set(transcriptomics['sample_id'].dropna())

common_sample_ids = sorted(met_ids & micro_ids & trans_ids)
print('Metabolomics samples:', len(met_ids))
print('Microbiota samples:', len(micro_ids))
print('Transcriptomics samples:', len(trans_ids))
print('Common samples across all 3:', len(common_sample_ids))

pd.Series(common_sample_ids, name='sample_id').to_csv(PROCESSED_DIR / 'common_sample_ids.csv', index=False)

metabolomics.to_csv(PROCESSED_DIR / 'metabolomics_clean.csv', index=False)
microbiota.to_csv(PROCESSED_DIR / 'microbiota_clean.csv', index=False)
transcriptomics.to_csv(PROCESSED_DIR / 'transcriptomics_clean.csv', index=False)
questionnaires.to_csv(PROCESSED_DIR / 'questionnaires_clean.csv', index=False)
weights.to_csv(PROCESSED_DIR / 'weights_clean.csv', index=False)

print('\nSaved cleaned CSVs to data/processed/')


Metabolomics samples: 235
Microbiota samples: 99
Transcriptomics samples: 89
Common samples across all 3: 78



Saved cleaned CSVs to data/processed/


# Compute velocity features and create datasets A, B, C

In [7]:
# Compute velocity features (matching PhD approach)
# Velocity = (later_timepoint - earlier_timepoint) / time_interval

# For simplicity, assuming equal time intervals (can adjust based on actual study timeline)
# WL phase: baseline (t1) -> post-WL (t2) 
# WM phase: post-WL (t2) -> post-WM (t4)

def compute_velocity_features(df, modality_name, feature_cols, meta_cols=['sample_id', 'participant_id', 'timepoint']):
    """
    Compute WL and WM velocity for each feature.
    Returns: DataFrame with [participant_id, WL_feature1, WL_feature2, ..., WM_feature1, WM_feature2, ...]
    """
    # Pivot to wide format: participant × (timepoint × features)
    participants = df['participant_id'].unique()
    
    velocity_data = []
    
    for pid in participants:
        participant_data = df[df['participant_id'] == pid].sort_values('timepoint')
        timepoints = participant_data['timepoint'].values
        
        # Check which timepoints are available
        has_t1 = 1 in timepoints
        has_t2 = 2 in timepoints
        has_t4 = 4 in timepoints
        
        row_data = {'participant_id': pid}
        
        # Compute WL velocity (t2 - t1) if both available
        if has_t1 and has_t2:
            t1_values = participant_data[participant_data['timepoint'] == 1][feature_cols].values[0]
            t2_values = participant_data[participant_data['timepoint'] == 2][feature_cols].values[0]
            wl_velocity = (t2_values - t1_values)  # Normalize by time if needed
            for i, feat in enumerate(feature_cols):
                row_data[f'WL_vel_{feat}'] = wl_velocity[i]
        else:
            # Missing data - set to NaN
            for feat in feature_cols:
                row_data[f'WL_vel_{feat}'] = np.nan
        
        # Compute WM velocity (t4 - t2) if both available
        if has_t2 and has_t4:
            t2_values = participant_data[participant_data['timepoint'] == 2][feature_cols].values[0]
            t4_values = participant_data[participant_data['timepoint'] == 4][feature_cols].values[0]
            wm_velocity = (t4_values - t2_values)
            for i, feat in enumerate(feature_cols):
                row_data[f'WM_vel_{feat}'] = wm_velocity[i]
        elif has_t1 and has_t4 and not has_t2:
            # Fallback: compute overall velocity if t2 is missing
            t1_values = participant_data[participant_data['timepoint'] == 1][feature_cols].values[0]
            t4_values = participant_data[participant_data['timepoint'] == 4][feature_cols].values[0]
            overall_velocity = (t4_values - t1_values) / 2  # Approximate as half for each phase
            for i, feat in enumerate(feature_cols):
                row_data[f'WM_vel_{feat}'] = overall_velocity[i]
        else:
            for feat in feature_cols:
                row_data[f'WM_vel_{feat}'] = np.nan
        
        velocity_data.append(row_data)
    
    velocity_df = pd.DataFrame(velocity_data)
    print(f"{modality_name} velocity features computed: {velocity_df.shape}")
    print(f"  Participants with complete WL velocity: {velocity_df[[c for c in velocity_df.columns if c.startswith('WL_vel')]].notna().all(axis=1).sum()}")
    print(f"  Participants with complete WM velocity: {velocity_df[[c for c in velocity_df.columns if c.startswith('WM_vel')]].notna().all(axis=1).sum()}")
    
    return velocity_df

# Compute velocities for each modality
met_meta_cols = ['sample_id', 'participant_id', 'timepoint', 'group']
met_features = [c for c in metabolomics.columns if c not in met_meta_cols]

micro_meta_cols = ['sample_id', 'participant_id', 'timepoint', 'Pair', 'group', 'Maintenancediet']
micro_features = [c for c in microbiota.columns if c not in micro_meta_cols]

trans_meta_cols = ['sample_id', 'participant_id', 'timepoint']
trans_features = [c for c in transcriptomics.columns if c not in trans_meta_cols]

print("="*60)
print("COMPUTING VELOCITY FEATURES")
print("="*60)

met_velocities = compute_velocity_features(metabolomics, "Metabolomics", met_features)
micro_velocities = compute_velocity_features(microbiota, "Microbiota", micro_features)
trans_velocities = compute_velocity_features(transcriptomics, "Transcriptomics", trans_features)

# Feature Selection for Transcriptomics (Dataset C is too large)
print("\n" + "="*60)
print("FEATURE SELECTION: TRANSCRIPTOMICS")
print("="*60)

# 1. Align transcriptomics with target
trans_fs_data = trans_velocities.merge(weights[['participant_id', 'absolute weight change ']], on='participant_id', how='inner')
trans_fs_data = trans_fs_data.dropna(subset=['absolute weight change '])

X_fs = trans_fs_data.drop(columns=['participant_id', 'absolute weight change '])
y_fs = trans_fs_data['absolute weight change ']

# 2. Train Random Forest for importance
imputer_fs = SimpleImputer(strategy='mean')
X_fs_imputed = imputer_fs.fit_transform(X_fs)

rf_fs = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_fs.fit(X_fs_imputed, y_fs)

# 3. Select top features
importance_df = pd.DataFrame({
    'feature': X_fs.columns,
    'importance': rf_fs.feature_importances_
}).sort_values('importance', ascending=False)

# Aggregate importance by gene to ensure we pick features measured in BOTH phases
importance_df['gene'] = importance_df['feature'].apply(lambda x: x.replace('WL_vel_', '').replace('WM_vel_', ''))
gene_importance = importance_df.groupby('gene')['importance'].sum().sort_values(ascending=False)

N_TOP_GENES = 100
top_genes = gene_importance.head(N_TOP_GENES).index.tolist()

selected_features = []
for g in top_genes:
    selected_features.append(f'WL_vel_{g}')
    selected_features.append(f'WM_vel_{g}')

trans_velocities_full = trans_velocities.copy()
trans_velocities = trans_velocities[['participant_id'] + selected_features]


print(f"Reduced transcriptomics to {len(selected_features)} features ({N_TOP_GENES} genes) based on aggregate RF importance")

# Compute velocities for anthropometrics
# Reshape questionnaires to long format first
anthro_features = ['k1_fatmass', 'k1_fatpercent', 'bmi1', 'weight1', 'k1_waistcircumference',
                  'k2_fatmass', 'k2_fatpercent', 'bmi2', 'weight2', 'k2_waistcircumference',
                  'k4_fatmass', 'k4_fatpercent', 'bmi4', 'weight4']
available_anthro = [c for c in anthro_features if c in questionnaires.columns]

def reshape_anthro_to_long(questionnaires_df, anthro_cols):
    """Reshape anthropometrics from wide (k1_fatmass, k2_fatmass, k4_fatmass) to long format"""
    # Create separate dataframes for each timepoint
    t1_cols = [c for c in anthro_cols if c.startswith('k1_') or c in ['bmi1', 'weight1']]
    t2_cols = [c for c in anthro_cols if c.startswith('k2_') or c in ['bmi2', 'weight2']]
    t4_cols = [c for c in anthro_cols if c.startswith('k4_') or c in ['bmi4', 'weight4']]
    
    def normalize_col_names(cols, tp):
        rename_map = {}
        for col in cols:
            new_col = col
            for prefix in ['k1_', 'k2_', 'k4_']:
                if new_col.startswith(prefix):
                    new_col = new_col[len(prefix):]
                    break
            if new_col.endswith('1') or new_col.endswith('2') or new_col.endswith('4'):
                new_col = new_col[:-1]
            rename_map[col] = new_col
        return rename_map
    
    t1_df = questionnaires_df[['participant_id'] + t1_cols].copy()
    t1_df = t1_df.rename(columns=normalize_col_names(t1_cols, 1))
    t1_df['timepoint'] = 1
    
    t2_df = questionnaires_df[['participant_id'] + t2_cols].copy()
    t2_df = t2_df.rename(columns=normalize_col_names(t2_cols, 2))
    t2_df['timepoint'] = 2
    
    t4_df = questionnaires_df[['participant_id'] + t4_cols].copy()
    t4_df = t4_df.rename(columns=normalize_col_names(t4_cols, 4))
    t4_df['timepoint'] = 4
    
    anthro_long = pd.concat([t1_df, t2_df, t4_df], ignore_index=True)
    return anthro_long

anthro_long = reshape_anthro_to_long(questionnaires, available_anthro)
anthro_feature_names = [c for c in anthro_long.columns if c not in ['participant_id', 'timepoint']]
anthro_velocities = compute_velocity_features(anthro_long, "Anthropometrics", anthro_feature_names)

# Create datasets A, B, C
print("\n" + "="*60)
print("CREATING DATASETS")
print("="*60)

# Dataset A: Full (Metabolomics + Microbiota + Anthropometrics)
# Participants with complete data in all three
participants_A = set(met_velocities['participant_id']) & set(micro_velocities['participant_id']) & set(anthro_velocities['participant_id'])
# Filter to those with minimal NaNs
met_A = met_velocities[met_velocities['participant_id'].isin(participants_A)]
micro_A = micro_velocities[micro_velocities['participant_id'].isin(participants_A)]
anthro_A = anthro_velocities[anthro_velocities['participant_id'].isin(participants_A)]

dataset_A = met_A.merge(micro_A, on='participant_id', how='inner').merge(anthro_A, on='participant_id', how='inner')
print(f"\nDataset A (Full): {len(dataset_A)} participants")

# Dataset B: Metabolomics-only + Anthropometrics
participants_B = set(met_velocities['participant_id']) & set(anthro_velocities['participant_id'])
met_B = met_velocities[met_velocities['participant_id'].isin(participants_B)]
anthro_B = anthro_velocities[anthro_velocities['participant_id'].isin(participants_B)]
dataset_B = met_B.merge(anthro_B, on='participant_id', how='inner')
print(f"Dataset B (Metabolomics-only): {len(dataset_B)} participants")

# Dataset C: Transcriptomics + Anthropometrics
participants_C = set(trans_velocities['participant_id']) & set(anthro_velocities['participant_id'])
trans_C = trans_velocities[trans_velocities['participant_id'].isin(participants_C)]
trans_C_full = trans_velocities_full[trans_velocities_full['participant_id'].isin(participants_C)]
anthro_C = anthro_velocities[anthro_velocities['participant_id'].isin(participants_C)]
dataset_C = trans_C.merge(anthro_C, on='participant_id', how='inner')
dataset_C_full = trans_C_full.merge(anthro_C, on='participant_id', how='inner')
print(f"Dataset C (Transcriptomics subset): {len(dataset_C)} participants")

# Merge with target
weights['participant_id'] = weights['participant_id'].apply(lambda x: f"{int(x):03d}" if pd.notna(x) else np.nan)

dataset_A = dataset_A.merge(weights[['participant_id', 'absolute weight change ']], on='participant_id', how='left')
dataset_B = dataset_B.merge(weights[['participant_id', 'absolute weight change ']], on='participant_id', how='left')
dataset_C = dataset_C.merge(weights[['participant_id', 'absolute weight change ']], on='participant_id', how='left')
dataset_C_full = dataset_C_full.merge(weights[['participant_id', 'absolute weight change ']], on='participant_id', how='left')

# Save datasets
dataset_A.to_csv(PROCESSED_DIR / 'dataset_A_full.csv', index=False)
dataset_B.to_csv(PROCESSED_DIR / 'dataset_B_metabolomics.csv', index=False)
dataset_C.to_csv(PROCESSED_DIR / 'dataset_C_transcriptomics.csv', index=False)
dataset_C_full.to_csv(PROCESSED_DIR / 'dataset_c_transcriptomics_full.csv', index=False)

print(f"\nDatasets saved to {PROCESSED_DIR}")
print(f"  dataset_A_full.csv: {dataset_A.shape}")
print(f"  dataset_B_metabolomics.csv: {dataset_B.shape}")
print(f"  dataset_C_transcriptomics.csv: {dataset_C.shape}")
print(f"  dataset_c_transcriptomics_full.csv: {dataset_C_full.shape}")

# Summary
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Dataset A: {len(dataset_A)} participants, {dataset_A.shape[1]-2} velocity features")
print(f"Dataset B: {len(dataset_B)} participants, {dataset_B.shape[1]-2} velocity features")
print(f"Dataset C: {len(dataset_C)} participants, {dataset_C.shape[1]-2} velocity features")
print(f"Dataset C (Full): {len(dataset_C_full)} participants, {dataset_C_full.shape[1]-2} velocity features")
print(f"\nTarget column: 'absolute weight change '")

dataset_A.head()
dataset_B.head()
dataset_C.head()

COMPUTING VELOCITY FEATURES
Metabolomics velocity features computed: (80, 195)
  Participants with complete WL velocity: 76
  Participants with complete WM velocity: 79


Microbiota velocity features computed: (34, 971)
  Participants with complete WL velocity: 32
  Participants with complete WM velocity: 32


Transcriptomics velocity features computed: (31, 39245)
  Participants with complete WL velocity: 27
  Participants with complete WM velocity: 31

FEATURE SELECTION: TRANSCRIPTOMICS


Reduced transcriptomics to 200 features (100 genes) based on aggregate RF importance


Anthropometrics velocity features computed: (82, 11)
  Participants with complete WL velocity: 82
  Participants with complete WM velocity: 0

CREATING DATASETS

Dataset A (Full): 33 participants
Dataset B (Metabolomics-only): 79 participants
Dataset C (Transcriptomics subset): 31 participants



Datasets saved to /home/bendavison/PycharmProjects/msc_dissertation/data/processed
  dataset_A_full.csv: (33, 1176)
  dataset_B_metabolomics.csv: (79, 206)
  dataset_C_transcriptomics.csv: (31, 212)
  dataset_c_transcriptomics_full.csv: (31, 39256)

SUMMARY
Dataset A: 33 participants, 1174 velocity features
Dataset B: 79 participants, 204 velocity features
Dataset C: 31 participants, 210 velocity features
Dataset C (Full): 31 participants, 39254 velocity features

Target column: 'absolute weight change '


,participant_id,WL_vel_TSPAN12_23554,WM_vel_TSPAN12_23554,WL_vel_PAOX_196743,WM_vel_PAOX_196743,WL_vel_RHAG_6005,WM_vel_RHAG_6005,WL_vel_TTTY10_246119,WM_vel_TTTY10_246119,WL_vel_GTSF1_121355,...,WL_vel_fatpercent,WL_vel_bmi,WL_vel_weight,WL_vel_waistcircumference,WM_vel_fatmass,WM_vel_fatpercent,WM_vel_bmi,WM_vel_weight,WM_vel_waistcircumference,absolute weight change
0,069,NaN,-11.988580,NaN,2.925860,NaN,1.472680,NaN,-0.266611,NaN,...,-3.7,-3.3,-8.6,-13.7,-2.7,-2.3,-0.9,-2.1,NaN,-2.664975
1,029,-0.901098,13.107316,-4.829018,12.171408,0.769322,1.132045,0.112470,0.762706,-0.006300,...,-3.4,-3.8,-10.2,-9.2,0.1,0.4,-0.1,-0.4,NaN,-0.522876
2,078,12.519338,7.590766,-18.301735,15.912144,1.291995,-0.365948,-1.320700,-0.039552,1.508697,...,-4.7,-4.8,-13.9,-10.2,2.3,1.3,0.8,2.4,NaN,2.316602
3,051,22.503197,15.503570,15.243261,0.418374,1.264861,-0.023799,7.869788,11.154812,-1.235983,...,-5.0,-5.8,-19.8,-20.0,0.1,-1.9,1.9,6.8,NaN,6.660137
4,088,-10.480613,-3.029857,-7.409769,18.079156,-1.168879,2.098232,35.050935,-14.094074,4.978197,...,-2.0,-5.9,-16.2,-14.8,-2.7,-1.5,-1.6,-4.4,NaN,-5.250597
